# Which engine and configuration made this file? Reading provenance

Every product the package writes carries a configuration stamp in a `.call` sidecar beside it, and
the attributed netCDF4 export copies that stamp into its global attributes together with the
name, size and SHA-256 of the file it was made from. This notebook writes two synthetic combined
products with different settings, reads each one's stamp and its attributed export, shows what
differs side by side, and checks that each attributed file came from the original it names.
Worked on fixed-seed synthetic data.

**Before this:** [06_read_products](../1_getting_started/06_read_products.ipynb) (Reading the output files). **Related:** [07_reproducible_runs](../2_processing/07_reproducible_runs.ipynb). **Needs:** core package only.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/7_analysis_and_outputs/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## Two synthetic products with different settings

The same Poisson counts are written twice by the package's combined-product writer. Each copy gets
its own configuration stamp: both as written by the v3.5 engine, the first under one reject
configuration and the second under another. The stamp is plain text in
`<file>.call`.

In [2]:
import tempfile
import numpy as np
import netCDF4
from arrayprobe import combine, attributed_export as ae
from arrayprobe.matlab_compat import datenum
from arrayprobe_uiops.decode_naming import combined_name

work = Path(tempfile.mkdtemp(prefix="nb40_"))
rng = np.random.default_rng(40)
N_TIME, N_BIN = 300, 32
edges = np.geomspace(0.01, 1.28, N_BIN + 1)          # mm
timevec = datenum(2020, 1, 1) + np.arange(N_TIME) / 86400.0
count = rng.poisson(5.0 * np.exp(-np.arange(N_BIN) / 10.0), size=(N_TIME, N_BIN)).astype(float)
nd = count / 1000.0

SETTINGS = {
    "A": ("3.5", "rej_hollow=0,max_edge_img_ar=4,max_comp_img_ar=4,area_ratio_z=0.5"),
    "B": ("3.5", "rej_hollow=1,max_edge_img_ar=6,max_comp_img_ar=5,area_ratio_z=0.5"),
}
products = {}
for label, (engine, reject) in SETTINGS.items():
    d = work / label
    d.mkdir()
    source = d / combined_name("base200101000000", "H", "2DS", engine)
    combine._write(source, 1, edges[:-1], edges[1:], (edges[:-1] + edges[1:]) / 2, np.diff(edges),
                   timevec, count * 0.01, count, nd, nd.sum(axis=1), nd.sum(axis=1) * 0.001)
    sd_stamp = (f"engine={engine} iat_file=2DS_intArrThreshold_MISSING.cdf profile=STOCK "
                f"reject=({reject}) shadow_diode=none default_int_arr_thresh=6e-06 (default)")
    Path(str(source) + ae.CALL_SUFFIX).write_text(
        f"engine={engine} combine=combine_sizedist.m inputs=sd.base200101000000.2DS.nc "
        f"inputs_built_with=({sd_stamp})")
    products[label] = source
    print(label, source.name, "+", source.name + ae.CALL_SUFFIX)

A base200101000000.combined.H.2DS.cdf + base200101000000.combined.H.2DS.cdf.call
B base200101000000.combined.H.2DS.cdf + base200101000000.combined.H.2DS.cdf.call


## Reading a stamp

`read_stamp` returns the sidecar's text, and `parse_stamp` splits it into keys. A nested value
such as `inputs_built_with` (the stamp of the stage before) is split again with `parse_nested`,
and the reject configuration with `parse_reject`.

In [3]:
stamps = {}
for label, source in products.items():
    top = ae.parse_stamp(ae.read_stamp(source))
    inner = ae.parse_nested(top["inputs_built_with"])
    stamps[label] = {"engine": top["engine"], "combine": top["combine"],
                     "profile": inner["profile"], **ae.parse_reject(inner["reject"])}
for label, s in stamps.items():
    print(label, s)

A {'engine': '3.5', 'combine': 'combine_sizedist.m', 'profile': 'STOCK', 'rej_hollow': '0', 'max_edge_img_ar': '4', 'max_comp_img_ar': '4', 'area_ratio_z': '0.5'}
B {'engine': '3.5', 'combine': 'combine_sizedist.m', 'profile': 'STOCK', 'rej_hollow': '1', 'max_edge_img_ar': '6', 'max_comp_img_ar': '5', 'area_ratio_z': '0.5'}


## The attributed export carries the stamp and the original's hash

`export` writes an analysis-ready netCDF4 file. Its global attributes name the source file, its
size and its SHA-256, and repeat the configuration. Attributes holding the creation time or the
package version are left out of the printout so it does not change between runs.

In [4]:
attributed = {label: ae.export(source, source.parent / "attributed.nc")
              for label, source in products.items()}
VOLATILE = ("date", "time", "version", "created", "history", "uuid", "id")
with netCDF4.Dataset(attributed["A"]) as ds:
    for name in ds.ncattrs():
        if name.startswith("arrayprobe_") and not any(v in name for v in VOLATILE):
            value = str(ds.getncattr(name))
            print(f"{name:40s} {value[:70]}")

arrayprobe_export_convention             arrayprobe-attributed-1.0
arrayprobe_export_module                 arrayprobe.attributed_export
arrayprobe_source_file                   base200101000000.combined.H.2DS.cdf
arrayprobe_source_sha256                 e929ff59b7b7278ffb8723f7a6301ec6cf14c30862e1fd59141debc6a2a0ccbc
arrayprobe_source_bytes                  239196
arrayprobe_source_writer                 ORACLES-variant combine_sizedist.m:202-231 (arrayprobe.combine._write)
arrayprobe_product_shape                 combined_sizedist
arrayprobe_missing_value_convention      Missing seconds are IEEE-754 NaN in the float64 payload, with no _Fill
arrayprobe_enhancements_applied          none
arrayprobe_enhancements_basis            none by construction when the producer is the port: 'the port is the f
arrayprobe_files_stitched                1
arrayprobe_tas_qc                        absent: the source carries no .tasqc.json, either because the run name
arrayprobe_call_stamp               

## What differs between the two files

The same keys read from each stamp, and the attributed files' provenance attributes, side by side.
Only rows that differ are marked.

In [5]:
def provenance(path):
    with netCDF4.Dataset(path) as ds:
        return {n: str(ds.getncattr(n)) for n in ds.ncattrs()
                if n.startswith("arrayprobe_") and not any(v in n for v in VOLATILE)}

prov = {label: provenance(p) for label, p in attributed.items()}
rows = [(k, stamps["A"][k], stamps["B"][k]) for k in stamps["A"]]
rows += [(k, prov["A"].get(k, ""), prov["B"].get(k, "")) for k in sorted(prov["A"])
         if k not in ("arrayprobe_source_sha256",)]
print(f"{'key':40s} {'A':>20s} {'B':>20s}")
for key, a, b in rows:
    if a != b:
        print(f"{key:40s} {a[:20]:>20s} {b[:20]:>20s}  <- differs")

key                                                         A                    B
rej_hollow                                                  0                    1  <- differs
max_edge_img_ar                                             4                    6  <- differs
max_comp_img_ar                                             4                    5  <- differs
arrayprobe_call_stamp                    engine=3.5 combine=c engine=3.5 combine=c  <- differs
arrayprobe_inputs_built_with             (engine=3.5 iat_file (engine=3.5 iat_file  <- differs


## Did this attributed file come from that original?

Hash the original with `sha256_of` and compare it with the `arrayprobe_source_sha256` the
attributed file carries, and the file size with `arrayprobe_source_bytes`. Changing a single
byte of the original breaks the match. The hash covers the file's bytes only; the stamp in the
sidecar is what records how those bytes were made.

In [6]:
for label in products:
    with netCDF4.Dataset(attributed[label]) as ds:
        recorded = ds.getncattr("arrayprobe_source_sha256")
        recorded_bytes = int(ds.getncattr("arrayprobe_source_bytes"))
    prov[label + "_hash"] = recorded
    actual = ae.sha256_of(products[label])
    print(label, "hash matches:", actual == recorded,
          "| size matches:", products[label].stat().st_size == recorded_bytes)

edited = work / "edited.cdf"
data = bytearray(products["A"].read_bytes())
data[-1] ^= 0xFF
edited.write_bytes(bytes(data))
print("edited copy matches A's recorded hash:", ae.sha256_of(edited) == prov["A_hash"])

A hash matches: True | size matches: True
B hash matches: True | size matches: True
edited copy matches A's recorded hash: False


## Summary

- A product's `.call` sidecar names the engine and configuration that wrote it; `read_stamp`,
  `parse_stamp`, `parse_nested` and `parse_reject` read it into keys.
- The attributed export repeats that configuration in its global attributes, with the source
  file's name, size and SHA-256.
- `sha256_of` on the original, compared with `arrayprobe_source_sha256`, checks that an attributed
  file was made from that exact file; any change to the original breaks the match.